In [ ]:
!pip install cellpose

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tiff
from cellpose import plot


class CalciumRecording:
    def __init__(self, path: str):
        self.data: np.ndarray = self._load(path)
        print(self.data.shape)
        self.path = path

    @staticmethod
    def _load_avi(path: str):
        cap = cv2.VideoCapture(path)

        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)

        cap.release()
        return np.array(frames)

    def _load(self, path: str):
        if "avi" in path:
            data = CalciumRecording._load_avi(path)
        else:
            data = tiff.imread(path)

        return data

    def visualize(self, masks: np.ndarray, flows: np.ndarray):
        T = self.data.shape[0]

        for t in range(T):
            fig = plt.figure(figsize=(20, 20))
            plot.show_segmentation(
                fig,
                self.data[t],
                masks[t],
                flows[t][0]
            )
            plt.show()



In [ ]:
from cellpose import models


class Segmenter:
    def __init__(self):
        pass

    @staticmethod
    def generate_mask(recording: CalciumRecording):
        model = models.CellposeModel(gpu=True)
        data = [recording.data[i] for i in range(recording.data.shape[0])]
        masks, flows, styles = model.eval(data, diameter=None, channels=[0, 0])
        return masks, flows


In [ ]:
import pickle
import numpy as np
import os

In [ ]:
from pathlib import Path

file_path = "/kaggle/input/datasets/mokarbalaee/neuroseg/Cropped Cb NLS gray.tif"
file_name = Path(file_path).name
recording: CalciumRecording = CalciumRecording(path=file_path)

In [ ]:
def save_results(masks, flows):
    np.save(f"masks+{file_name}.npy", masks)
    with open(f"flows+{file_name}.pkl", "wb") as f:
        pickle.dump(flows, f)

def load_results():
    masks = np.load(f"masks+{file_name}.npy", allow_pickle=True)
    with open(f"flows+{file_name}.pkl", "rb") as f:
        flows = pickle.load(f)
    return masks, flows

In [ ]:
if os.path.exists(f"masks+{file_name}.npy"):
    masks, flows = load_results()
else:
    masks, flows = Segmenter.generate_mask(recording)
    save_results(masks, flows)


print(len(masks))
print(len(flows))

In [ ]:
recording.visualize(masks, flows)